<a href="https://colab.research.google.com/github/K-Siddharth-Reddy-CR7/Face-Mask-Detection-/blob/main/test_finalmp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow opencv-python matplotlib tf-explain scikit-learn

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import zipfile

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

!pip install lime
from lime import lime_image
from skimage.segmentation import mark_boundaries
import shap

In [ ]:
import zipfile

zip_path = "archive.zip"  # change if needed, ensure this file is uploaded to your Colab environment

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("dataset")

In [ ]:
import os

for root, dirs, files in os.walk("dataset"):
    print(root)

In [ ]:
data = []
labels = []

categories = ["with_mask", "without_mask"]
path = "dataset/data"

for category in categories:
    folder_path = os.path.join(path, category)
    label = categories.index(category)

    for img in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img)
        image = cv2.imread(img_path)

        if image is not None:
            image = cv2.resize(image, (100,100))
            data.append(image)
            labels.append(label)

data = np.array(data) / 255.0
labels = np.array(labels)

print("Total images:", len(data))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.2, random_state=42
)

In [ ]:
model = Sequential()

model.add(Conv2D(32,(3,3),activation='relu',input_shape=(100,100,3)))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dense(2,activation='softmax'))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_test, y_test)
)

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy:", acc)

In [ ]:
model.save("mask_detector_model.h5")

In [ ]:
!wget https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt

!wget https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel

In [ ]:
import cv2

prototxt = "deploy.prototxt"
weights = "res10_300x300_ssd_iter_140000.caffemodel"

faceNet = cv2.dnn.readNetFromCaffe(prototxt, weights)

In [ ]:
def detect_faces(image):
    (h, w) = image.shape[:2]
    blob = cv2.dnn.blobFromImage(cv2.resize(image, (300, 300)), 1.0,
                                 (300, 300), (104.0, 177.0, 123.0))
    faceNet.setInput(blob)
    detections = faceNet.forward()

    faces = []
    boxes = []

    for i in range(0, detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > 0.5: # Confidence threshold
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            (startX, startY, endX, endY) = box.astype("int")

            face = image[startY:endY, startX:endX]
            if face.shape[0] > 0 and face.shape[1] > 0: # Ensure face is not empty
                face = cv2.resize(face, (100, 100))
                faces.append(face)
                boxes.append((startX, startY, endX, endY))
    return faces, boxes

img = cv2.imread("mixgrp.jpeg")

if img is None:
    print("Error: Image not found")
else:
    faces, boxes = detect_faces(img)

    print("Faces detected:", len(faces))

    class_names = ["With Mask", "Without Mask"]

    for face, box in zip(faces, boxes):
        (x1,y1,x2,y2) = box

        face_array = face.astype("float32") / 255.0
        face_array = np.reshape(face_array,(1,100,100,3))

        pred = model.predict(face_array)
        cls = np.argmax(pred)
        conf = np.max(pred)*100

        label = class_names[cls]
        color = (0,255,0) if cls==0 else (0,0,255)

        cv2.putText(img,f"{label} ({conf:.1f}%)",(x1,y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,0.6,color,2)
        cv2.rectangle(img,(x1,y1),(x2,y2),color,2)

    plt.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
    plt.axis("off")

In [ ]:
import tensorflow as tf

if len(faces) == 0:
    print("No faces for Grad-CAM")
else:
    for i, (face, box) in enumerate(zip(faces, boxes)):
        face_array = face.astype("float32") / 255.0
        img_array = np.reshape(face_array,(1,100,100,3))

        img_tensor = tf.convert_to_tensor(img_array)

        last_conv = None
        for layer in reversed(model.layers):
            if "conv" in layer.name:
                last_conv = layer
                break

        conv_model = tf.keras.Model(model.inputs, last_conv.output)

        classifier_input = tf.keras.Input(shape=last_conv.output.shape[1:])
        x = classifier_input

        start=False
        for layer in model.layers:
            if start:
                x = layer(x)
            if layer.name == last_conv.name:
                start=True

        classifier_model = tf.keras.Model(classifier_input, x)

        with tf.GradientTape() as tape:
            conv_out = conv_model(img_tensor)
            tape.watch(conv_out)
            preds = classifier_model(conv_out)
            loss = preds[:, np.argmax(preds[0])]

        grads = tape.gradient(loss, conv_out)
        pooled = tf.reduce_mean(grads, axis=(0,1,2))

        conv_out = conv_out[0]
        heatmap = conv_out @ pooled[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)

        heatmap = tf.maximum(heatmap,0)/tf.reduce_max(heatmap)
        heatmap = heatmap.numpy()

        heatmap = cv2.resize(heatmap,(100,100))
        heatmap = np.uint8(255*heatmap)

        heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

        face_uint8 = (face_array*255).astype("uint8")
        overlay = cv2.addWeighted(face_uint8,0.6,heatmap,0.4,0)

        plt.figure()
        plt.imshow(overlay)
        plt.title(f"Grad-CAM for Face {i+1}")
        plt.axis("off")
    plt.show()

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

if len(faces) == 0:
    print("No faces for LIME")
else:
    def predict_fn(images):
        images = np.array(images)
        images = np.array([cv2.resize(img,(100,100)) for img in images])
        images = images.astype("float32")/255.0
        return model.predict(images)

    explainer = lime_image.LimeImageExplainer()

    for i, face in enumerate(faces):
        explanation = explainer.explain_instance(
            face.astype('double'),
            predict_fn,
            top_labels=2,
            hide_color=0,
            num_samples=1000
        )

        temp,mask = explanation.get_image_and_mask(
            explanation.top_labels[0],
            positive_only=True,
            num_features=5,
            hide_rest=False
        )

        plt.figure()
        plt.imshow(mark_boundaries(temp/255.0,mask))
        plt.title(f"LIME Explanation for Face {i+1}")
        plt.axis("off")
    plt.show()

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Select background images from training data
background_size = 30
background = X_train[np.random.choice(X_train.shape[0], background_size, replace=False)]

# Step 2: Create SHAP explainer
explainer = shap.DeepExplainer(model, background)

# Step 3: Generate SHAP values for the test image
shap_values = explainer.shap_values(img_array)

# Step 4: Display SHAP explanation
plt.figure(figsize=(6,6))
shap.image_plot(shap_values, img_array)

In [ ]:
predicted_class = np.argmax(model.predict(img_array))

plt.figure(figsize=(6,6))
shap.image_plot([shap_values[:, :, :, :, predicted_class]], img_array)